# Lesson 1.3: Space Complexity and Python Costs

RENDUS Technologies | DSA with Python | Module 1

Run each cell from top to bottom (Runtime > Run all). Try to predict the output before you run a cell.

## Time and space of 5 short functions

`space_complexity_examples.py`

In [ ]:
"""Lesson 1.3 - Practice: time and space complexity of 5 short functions.

Space means AUXILIARY space: extra memory beyond the input itself.
"""

import sys


def sum_list(nums):
    """Time O(n), space O(1): a single running total."""
    total = 0
    for x in nums:
        total += x
    return total


def doubled(nums):
    """Time O(n), space O(n): builds a brand new list of n items."""
    return [x * 2 for x in nums]


def make_grid(n):
    """Time O(n^2), space O(n^2): an n x n grid."""
    return [[0] * n for _ in range(n)]


def factorial(n):
    """Time O(n), space O(n): n calls wait on the call stack."""
    if n <= 1:
        return 1
    return n * factorial(n - 1)


def reverse_in_place(nums):
    """Time O(n), space O(1): swaps inside the input, no new list."""
    left, right = 0, len(nums) - 1
    while left < right:
        nums[left], nums[right] = nums[right], nums[left]
        left += 1
        right -= 1
    return nums


if __name__ == "__main__":
    print("sum_list([1,2,3])        =", sum_list([1, 2, 3]))
    print("doubled([1,2,3])         =", doubled([1, 2, 3]))
    print("make_grid(3)             =", make_grid(3))
    print("factorial(5)             =", factorial(5))
    print("reverse_in_place([1,2,3])=", reverse_in_place([1, 2, 3]))

    print("\nRecursion depth is space. Default limit:", sys.getrecursionlimit())
    try:
        factorial(5000)
    except RecursionError:
        print("factorial(5000) -> RecursionError (stack too deep)")

    print("""
Summary
  function          time     space
  sum_list          O(n)     O(1)
  doubled           O(n)     O(n)
  make_grid         O(n^2)   O(n^2)
  factorial         O(n)     O(n)   (call stack)
  reverse_in_place  O(n)     O(1)""")

## pop(0) vs deque.popleft()

`pop_zero_vs_deque.py`

In [ ]:
"""Lesson 1.3 - Practice: replace a slow pop(0) loop with deque.popleft()."""

import time
from collections import deque

N = 100_000


def drain_with_list(n):
    """list.pop(0) shifts every remaining element: O(n) each, O(n^2) total."""
    queue = list(range(n))
    while queue:
        queue.pop(0)


def drain_with_deque(n):
    """deque.popleft() is O(1) each, O(n) total."""
    queue = deque(range(n))
    while queue:
        queue.popleft()


def timed(func, n):
    start = time.perf_counter()
    func(n)
    return time.perf_counter() - start


if __name__ == "__main__":
    t_list = timed(drain_with_list, N)
    t_deque = timed(drain_with_deque, N)
    print(f"Removing {N:,} items from the front")
    print(f"  list.pop(0)      : {t_list:.4f} s")
    print(f"  deque.popleft()  : {t_deque:.4f} s")
    print(f"  deque is about {t_list / t_deque:,.0f}x faster")

## in list vs in set

`list_vs_set_timing.py`

In [ ]:
"""Lesson 1.3 - Speed demonstration: `x in list` versus `x in set`.

The lesson idea: check membership many times in 100,000 numbers.
A full 100,000 checks on the list takes minutes, so CHECKS defaults to a
smaller number. Both sides use the same number of checks, so the comparison
is fair. Raise CHECKS if you want to wait for the full demo.
"""

import random
import time

N = 100_000        # size of the data
CHECKS = 2_000     # how many membership checks to run (try 100_000 for the full demo)


def time_checks(container, queries):
    start = time.perf_counter()
    found = 0
    for q in queries:
        if q in container:
            found += 1
    return time.perf_counter() - start, found


if __name__ == "__main__":
    random.seed(1)
    data_list = list(range(N))
    data_set = set(data_list)
    # Worst case for the list: values near the end, or missing (full scan)
    queries = [random.randint(0, 2 * N) for _ in range(CHECKS)]

    t_list, found_list = time_checks(data_list, queries)
    t_set, found_set = time_checks(data_set, queries)
    assert found_list == found_set

    print(f"{CHECKS:,} membership checks in {N:,} numbers")
    print(f"  list : {t_list:.4f} s   (O(n) per check)")
    print(f"  set  : {t_set:.6f} s   (O(1) average per check)")
    print(f"  set is about {t_list / t_set:,.0f}x faster")

## s += c vs collect and join

`string_building_join.py`

In [ ]:
"""Lesson 1.3 - Building strings: `s += c` in a loop versus collect and join.

Strings are immutable, so `+=` may copy the whole string on every step
(up to O(n^2) total). Collecting pieces in a list and calling "".join once
is O(n) overall.

Note: CPython sometimes optimises `s += c`, so the gap can look small on your
machine. Do not rely on that optimisation, and it does not exist in many
other Python implementations or when the string is referenced elsewhere.
"""

import time

N = 200_000


def build_with_plus(n):
    result = ""
    for _ in range(n):
        result += "a"
    return result


def build_with_join(n):
    parts = []
    for _ in range(n):
        parts.append("a")
    return "".join(parts)


def timed(func, n):
    start = time.perf_counter()
    out = func(n)
    return time.perf_counter() - start, out


if __name__ == "__main__":
    t_plus, a = timed(build_with_plus, N)
    t_join, b = timed(build_with_join, N)
    assert a == b
    print(f"Building a string of {N:,} characters")
    print(f"  s += c      : {t_plus:.4f} s")
    print(f"  list + join : {t_join:.4f} s")

## Three ways to reverse a string

`reverse_string_three_ways.py`

In [ ]:
"""Lesson 1.3 - Three ways to reverse a string.

All three take O(n) time. Strings are immutable, so any method that returns
a string needs O(n) space. The two-pointer idea works in O(1) extra space
beyond the list of characters it swaps inside.
"""


def reverse_slicing(s):
    """Python builds a reversed copy. Extra space O(n)."""
    return s[::-1]


def reverse_join(s):
    """Iterate backwards and join. Extra space O(n)."""
    return "".join(reversed(s))


def reverse_two_pointers(s):
    """Swap from both ends on a list of characters. O(1) beyond that list."""
    chars = list(s)
    left, right = 0, len(chars) - 1
    while left < right:
        chars[left], chars[right] = chars[right], chars[left]
        left += 1
        right -= 1
    return "".join(chars)


if __name__ == "__main__":
    tests = ["hello", "", "a", "ab", "racecar", "Python 3"]
    for s in tests:
        a = reverse_slicing(s)
        b = reverse_join(s)
        c = reverse_two_pointers(s)
        assert a == b == c == s[::-1]
        print(repr(s), "->", repr(a))